# Kaggle Inference Notebook (Qwen3-4B)

This notebook runs **inference only** for the `llm-classification-finetuning` competition using the repository's existing model CLI and checkpoint loading path (`uv run model qwen3-4b --test` equivalent).

It does **not** retrain or rebuild the classifier head.

## Required Kaggle inputs

Attach these datasets/inputs to the notebook:

1. **Competition data** (`test.csv`, `sample_submission.csv`) mounted under `/kaggle/input/...`.
2. **Repository source snapshot** containing `pyproject.toml`, `config.yaml`, and `src/llm_classification_finetuning/`.
3. **Trained head checkpoint** file (`head.pt` or `head_<tag>.pt`) either inside that source snapshot (`models/qwen3_4b/`) or in another mounted dataset.

No Kaggle API token is required for this inference path because all files are read from mounted inputs.

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

MODEL_SELECTOR = 'qwen3-4b'
CHECKPOINT_TAG = None  # e.g. 'config_inf' if using head_config_inf.pt

# Optional hints when auto-discovery finds multiple candidates.
COMPETITION_DIR_HINT = None  # e.g. '/kaggle/input/llm-classification-finetuning'
REPO_DATASET_DIR_HINT = None  # e.g. '/kaggle/input/llm-classification-finetuning-repo'
CHECKPOINT_DATASET_DIR_HINT = None  # e.g. '/kaggle/input/llm-classification-checkpoint'

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_WORKDIR = KAGGLE_WORKING / 'llm-classification-finetuning'
OUTPUT_SUBMISSION_PATH = KAGGLE_WORKING / 'submission.csv'

REQUIRED_SUBMISSION_COLUMNS = [
    'id',
    'winner_model_a',
    'winner_model_b',
    'winner_tie',
]


In [ ]:
def _resolve_hint_dir(value: str | None, label: str) -> Path | None:
    if value is None:
        return None
    path = Path(value).expanduser().resolve()
    if not path.is_dir():
        raise FileNotFoundError(f'{label} hint is not a directory: {path}')
    return path


def _find_competition_dir(hint: Path | None) -> Path:
    if hint is not None:
        candidates = [hint]
    else:
        candidates = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
    matches: list[Path] = []
    for candidate in candidates:
        if (candidate / 'test.csv').is_file() and (candidate / 'sample_submission.csv').is_file():
            matches.append(candidate)
    if not matches:
        raise FileNotFoundError(
            'Could not find competition files (test.csv + sample_submission.csv) under /kaggle/input. '
            'Attach the competition dataset or set COMPETITION_DIR_HINT.'
        )
    if len(matches) > 1 and hint is None:
        options = '\n'.join(str(path) for path in matches)
        raise RuntimeError(
            'Found multiple competition-like inputs. Set COMPETITION_DIR_HINT to one of:\n' + options
        )
    return matches[0]


def _find_repo_source_dir(hint: Path | None) -> Path:
    if hint is not None:
        candidates = [hint]
    else:
        candidates = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
    matches: list[Path] = []
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'llm_classification_finetuning').is_dir():
            matches.append(candidate)
    if not matches:
        raise FileNotFoundError(
            'Could not find repository source input with pyproject.toml and src/llm_classification_finetuning. '
            'Attach a repository snapshot dataset or set REPO_DATASET_DIR_HINT.'
        )
    if len(matches) > 1 and hint is None:
        options = '\n'.join(str(path) for path in matches)
        raise RuntimeError(
            'Found multiple repository-like inputs. Set REPO_DATASET_DIR_HINT to one of:\n' + options
        )
    return matches[0]


def _find_checkpoint_file(*, tag: str | None, repo_dir: Path, checkpoint_hint: Path | None) -> Path:
    checkpoint_name = f'head_{tag}.pt' if tag else 'head.pt'

    candidate_paths: list[Path] = [
        repo_dir / 'models' / 'qwen3_4b' / checkpoint_name,
    ]
    if checkpoint_hint is not None:
        candidate_paths.extend(
            [
                checkpoint_hint / 'models' / 'qwen3_4b' / checkpoint_name,
                checkpoint_hint / checkpoint_name,
            ]
        )

    for dataset_dir in KAGGLE_INPUT.iterdir():
        if not dataset_dir.is_dir():
            continue
        candidate_paths.extend(
            [
                dataset_dir / 'models' / 'qwen3_4b' / checkpoint_name,
                dataset_dir / checkpoint_name,
            ]
        )

    for candidate in candidate_paths:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f'Could not find {checkpoint_name}. Attach a dataset containing this checkpoint or set CHECKPOINT_DATASET_DIR_HINT.'
    )


In [ ]:
competition_dir = _find_competition_dir(_resolve_hint_dir(COMPETITION_DIR_HINT, 'competition'))
repo_source_dir = _find_repo_source_dir(_resolve_hint_dir(REPO_DATASET_DIR_HINT, 'repository'))
checkpoint_source = _find_checkpoint_file(
    tag=CHECKPOINT_TAG,
    repo_dir=repo_source_dir,
    checkpoint_hint=_resolve_hint_dir(CHECKPOINT_DATASET_DIR_HINT, 'checkpoint'),
)

print(f'Competition input: {competition_dir}')
print(f'Repository input: {repo_source_dir}')
print(f'Checkpoint source: {checkpoint_source}')

if PROJECT_WORKDIR.exists():
    shutil.rmtree(PROJECT_WORKDIR)
shutil.copytree(repo_source_dir, PROJECT_WORKDIR)

raw_dir = PROJECT_WORKDIR / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
for filename in ('test.csv', 'sample_submission.csv'):
    source = competition_dir / filename
    if not source.is_file():
        raise FileNotFoundError(f'Missing required competition file: {source}')
    shutil.copy2(source, raw_dir / filename)

checkpoint_name = f"head_{CHECKPOINT_TAG}.pt" if CHECKPOINT_TAG else 'head.pt'
checkpoint_destination = PROJECT_WORKDIR / 'models' / 'qwen3_4b' / checkpoint_name
checkpoint_destination.parent.mkdir(parents=True, exist_ok=True)
if checkpoint_source.resolve() != checkpoint_destination.resolve():
    shutil.copy2(checkpoint_source, checkpoint_destination)

print(f'Working project directory: {PROJECT_WORKDIR}')
print(f'Raw competition files staged in: {raw_dir}')
print(f'Checkpoint available at: {checkpoint_destination}')


In [ ]:
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
if gpu_count < 1:
    raise RuntimeError(
        "No CUDA GPU detected. This repository's model pipeline requires CUDA devices for inference. "
        'Enable a GPU accelerator in Kaggle and rerun.'
    )

selected_devices = [f'cuda:{index}' for index in range(min(2, gpu_count))]

base_config_path = PROJECT_WORKDIR / 'config.yaml'
with base_config_path.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

config['gpu']['devices'] = selected_devices
config['data']['raw_dir'] = 'data/raw'
config['data']['processed_path'] = 'data/processed/train_folds.parquet'

kaggle_config_path = PROJECT_WORKDIR / 'config.kaggle.yaml'
with kaggle_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print(f'Detected CUDA device count: {gpu_count}')
print(f'Configured gpu.devices: {selected_devices}')
print(f'Wrote Kaggle config: {kaggle_config_path}')


In [ ]:
project_src = PROJECT_WORKDIR / 'src'
if str(project_src) not in sys.path:
    sys.path.insert(0, str(project_src))

os.chdir(PROJECT_WORKDIR)

from llm_classification_finetuning.models.cli import run as run_model_cli

args = [MODEL_SELECTOR, '--test', '--config', str(kaggle_config_path)]
if CHECKPOINT_TAG:
    args.extend(['--checkpoint-tag', CHECKPOINT_TAG])

exit_code = run_model_cli(args)
if exit_code != 0:
    raise RuntimeError(f'Model CLI exited with code {exit_code}.')

test_results_dir = PROJECT_WORKDIR / 'results' / 'test'
submission_candidates = sorted(
    test_results_dir.glob(f'{MODEL_SELECTOR}-*/submission.csv'),
    key=lambda path: path.stat().st_mtime,
)
if not submission_candidates:
    raise FileNotFoundError(f'No submission.csv found under {test_results_dir}')

generated_submission_path = submission_candidates[-1]
shutil.copy2(generated_submission_path, OUTPUT_SUBMISSION_PATH)
print(f'Generated submission: {generated_submission_path}')
print(f'Copied final submission to: {OUTPUT_SUBMISSION_PATH}')


In [ ]:
submission = pd.read_csv(OUTPUT_SUBMISSION_PATH)
if list(submission.columns) != REQUIRED_SUBMISSION_COLUMNS:
    raise ValueError(
        f'Submission columns must be exactly {REQUIRED_SUBMISSION_COLUMNS}; got {list(submission.columns)}'
    )

sample_submission_path = PROJECT_WORKDIR / 'data' / 'raw' / 'sample_submission.csv'
if sample_submission_path.is_file():
    sample_submission = pd.read_csv(sample_submission_path)
    if not np.array_equal(sample_submission['id'].to_numpy(), submission['id'].to_numpy()):
        raise ValueError('Submission IDs do not exactly match sample_submission.csv IDs in order.')

probabilities = submission[REQUIRED_SUBMISSION_COLUMNS[1:]].to_numpy(dtype=np.float64)
if not np.isfinite(probabilities).all():
    raise ValueError('Submission probabilities contain non-finite values.')
if ((probabilities < 0.0) | (probabilities > 1.0)).any():
    raise ValueError('Submission probabilities must be in [0, 1].')

row_sums = probabilities.sum(axis=1)
if not np.allclose(row_sums, 1.0, rtol=1e-6, atol=1e-6):
    max_error = float(np.max(np.abs(row_sums - 1.0)))
    raise ValueError(f'Submission probability rows must sum to 1. Max absolute error: {max_error:.3e}')

print(f'Validated {len(submission):,} rows.')
print(f'Final Kaggle submission path: {OUTPUT_SUBMISSION_PATH}')
submission.head()


## Output

Submit `/kaggle/working/submission.csv` to the competition.

The full run artifacts remain in `/kaggle/working/llm-classification-finetuning/results/test/`.